# Simulate Tax for all Portfolioa

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pandas.tseries.offsets as pd_offsets
import pickle
import plotly.graph_objects as go
from typing import Dict, Tuple
from dateutil.relativedelta import relativedelta
import itertools

In [ ]:
from utils.plots import (
    draw_growth_chart,
    draw_telltale_chart,
    draw_risk_reward_chart,
    draw_periodic_return,
)
from utils.plots import (
    draw_correlations,
    compare_portfolios,
    draw_max_portfolio_drawdowns,
    draw_min_portfolio_returns,
)
from utils.math import (
    gmean,
    calc_min_returns,
    calc_max_drawdown,
    calc_correlations_over_time,
    normalize,
    calc_returns,
)
from utils.math.cagr import cagr
from utils.math import to_float, calc_growth, normalize_df
from utils.data import cached, read_csv
from utils.portfolio import Portfolio, Asset, GermanTaxModel, MAPortfolio

In [ ]:
clean_data_path = Path("clean_data")
cache_path = Path("cached_data")

In [ ]:
input_path = clean_data_path / "etfs.xlsx"
etfs = pd.read_excel(input_path, index_col=0)
etfs.index = pd.to_datetime(etfs.index)
etfs["cash"] = 100.0
etfs

In [ ]:
import plotly.express as px

window = 290

sp500_ta = etfs["1x_sp500_us"].to_frame()

sp500_ta["ma"] = sp500_ta["1x_sp500_us"].rolling(window=window).mean()
sp500_ta["std"] = sp500_ta["1x_sp500_us"].rolling(window=window).std()
sp500_ta["bb_upper"] = sp500_ta["ma"] + 2 * sp500_ta["std"]
sp500_ta["bb_lower"] = sp500_ta["ma"] - 2 * sp500_ta["std"]

# sp500_ta
sp500_ta = sp500_ta.loc["2020-01-01":"2022-01-01"]

In [ ]:
import plotly.graph_objects as go

price = sp500_ta["1x_sp500_us"]
ma = sp500_ta["ma"]

cross_up = (price > ma) & (price.shift(1) <= ma.shift(1))
cross_down = (price < ma) & (price.shift(1) >= ma.shift(1))

cross_up_dates = sp500_ta.index[cross_up]
cross_down_dates = sp500_ta.index[cross_down]

# Build figure as before
fig = go.Figure(
    [
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["bb_upper"],
            line=dict(color="rgba(255,0,0,0.3)", width=0),
            showlegend=False,
            name="BB Upper",
        ),
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["bb_lower"],
            line=dict(color="rgba(255,0,0,0.3)", width=0),
            fill="tonexty",
            fillcolor="rgba(200,0,0,0.2)",
            showlegend=True,
            name="Bollinger Band",
        ),
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["1x_sp500_us"],
            line=dict(color="blue"),
            name="Price",
        ),
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["ma"],
            line=dict(color="red", width=2),
            name="Moving Average",
        ),
    ]
)

# Add vertical lines for cross_up (green) and cross_down (red)
shapes = []
for date in cross_up_dates:
    shapes.append(
        dict(
            type="line",
            xref="x",
            yref="paper",
            x0=date,
            x1=date,
            y0=0,
            y1=1,
            line=dict(color="green", width=2, dash="dot"),
        )
    )
for date in cross_down_dates:
    shapes.append(
        dict(
            type="line",
            xref="x",
            yref="paper",
            x0=date,
            x1=date,
            y0=0,
            y1=1,
            line=dict(color="red", width=2, dash="dot"),
        )
    )

fig.update_layout(
    title="SP500 with Moving Average and Shaded Bollinger Bands", shapes=shapes
)
fig.show()


In [ ]:
p_sp500 = Portfolio(
    {
        "1x_sp500_eu": 100.0,
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)


p_2x_sp500 = Portfolio(
    {
        "2x_sp500_eu": 100.0,
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)

p_2x_sp500_ma = MAPortfolio(
    {
        "2x_sp500_eu": dict(dist=100, ma=290, ma_asset="1x_sp500_eu"),
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)

p_2x_ndx100_ma = MAPortfolio(
    {
        "2x_ndx100_eu": dict(dist=100, ma=310, ma_asset="1x_ndx100_eu"),
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)

In [ ]:
p_2x_ndx100_2x_sp500_ma = MAPortfolio(
    {
        "2x_sp500_eu": dict(dist=80, ma=290, ma_asset="1x_sp500_eu"),
        "2x_ndx100_eu": dict(dist=20, ma=310, ma_asset="1x_ndx100_eu"),
    },
    start_value=1000,
    rebalancing=relativedelta(months=3),
    rebalancing_offset=relativedelta(days=-8),
    spread=0.002,
    tax_model=GermanTaxModel(),
).backtest(etfs)

p_2x_ndx100_2x_sp500_ma_sliced = p_2x_ndx100_2x_sp500_ma.loc["1945":"2025"]

In [ ]:
# import optuna
# from optuna.samplers import TPESampler
# from dateutil.relativedelta import relativedelta
# import optunahub
# import numpy as np

# optuna.logging.set_verbosity(optuna.logging.WARNING)
# n_train_iter = 100
# module = optunahub.load_module(package="samplers/auto_sampler")

# study = optuna.create_study(direction="maximize", sampler=module.AutoSampler())


# def get_all_metrics(study, new_cagr=None, new_drawdown=None):
#     cagr_list = [
#         t.user_attrs.get("cagr")
#         for t in study.trials
#         if t.user_attrs.get("cagr") is not None
#     ]
#     drawdown_list = [
#         t.user_attrs.get("max_drawdown")
#         for t in study.trials
#         if t.user_attrs.get("max_drawdown") is not None
#     ]
#     # Add current metrics
#     if new_cagr is not None and new_drawdown is not None:
#         cagr_list.append(new_cagr)
#         drawdown_list.append(new_drawdown)
#     return cagr_list, drawdown_list


# def dynamic_min_max_scaler(value, values):
#     min_val = min(values)
#     max_val = max(values)
#     if max_val == min_val:
#         return 0.5  # Neutral in case all values are the same
#     return max(0.0, min(1.0, (value - min_val) / (max_val - min_val)))


# def objective(trial):
#     ma_sp500 = trial.suggest_int("ma_sp500", 200, 350)
#     ma_ns100 = trial.suggest_int("ma_ns100", 200, 350)

#     # Enforce their sum is 100
#     p_sp500 = trial.suggest_int("p_sp500", 1, 99)
#     p_ns100 = 100 - p_sp500

#     p = MAPortfolio(
#         {
#             "2x_sp500_eu": dict(dist=p_sp500, ma=ma_sp500, ma_asset="1x_sp500_eu"),
#             "2x_ndx100_eu": dict(dist=p_ns100, ma=ma_ns100, ma_asset="1x_ndx100_eu"),
#         },
#         start_value=1000,
#         rebalancing=relativedelta(months=3),
#         rebalancing_offset=relativedelta(days=-8),
#         spread=0.002,
#         tax_model=GermanTaxModel(),
#     )
#     result = p.backtest(etfs)
#     result = result.loc["1945":"2025"]

#     my_cagr = cagr(result)
#     max_drawdown = calc_max_drawdown(result)[0]["sum"].item() * -1

#     WEIGHT_CAGR = 0.7
#     WEIGHT_DRAWDOWN = 0.3

#     cagr_list, drawdown_list = get_all_metrics(study, my_cagr, max_drawdown)

#     cagr_norm = dynamic_min_max_scaler(my_cagr, cagr_list)
#     drawdown_norm = dynamic_min_max_scaler(max_drawdown, drawdown_list)

#     score = (WEIGHT_CAGR * cagr_norm) + (WEIGHT_DRAWDOWN * (1 - drawdown_norm))

#     trial.set_user_attr("cagr", my_cagr)
#     trial.set_user_attr("max_drawdown", max_drawdown)
#     trial.set_user_attr(
#         "score_components",
#         {
#             "cagr_norm": cagr_norm,
#             "drawdown_norm": drawdown_norm,
#         },
#     )

#     return score


# study.optimize(objective, n_trials=150, show_progress_bar=True)

# completed_trials = [
#     t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
# ]


# def is_pareto_efficient(costs):
#     is_efficient = np.ones(costs.shape[0], dtype=bool)
#     for i, c in enumerate(costs):
#         if is_efficient[i]:
#             # Keep if not dominated by any
#             is_efficient[is_efficient] = np.any(
#                 costs[is_efficient] < c, axis=1
#             ) | np.all(costs[is_efficient] == c, axis=1)
#             is_efficient[i] = True  # Keep self
#     return is_efficient


# def pareto_front(trials):
#     # We want high cagr, low drawdown
#     data = np.array(
#         [
#             [t.user_attrs.get("cagr", -1e10), t.user_attrs.get("max_drawdown", 1e10)]
#             for t in trials
#         ]
#     )
#     pareto = []
#     for i, (cagr, dd) in enumerate(data):
#         nondominated = True
#         for j, (other_cagr, other_dd) in enumerate(data):
#             if j != i:
#                 if (other_cagr >= cagr and other_dd <= dd) and (
#                     other_cagr > cagr or other_dd < dd
#                 ):
#                     nondominated = False
#                     break
#         if nondominated:
#             pareto.append(i)
#     return np.array(trials)[pareto]


# pareto_trials = pareto_front(completed_trials)
# print("\n=== Pareto Front (Highest CAGR, Lowest Drawdown) ===")
# for t in pareto_trials:
#     print(f"\nTrial #{t.number}")
#     print(f"  Value (Score): {t.value}")
#     print(f"  Params: {t.params}")
#     print(f"  CAGR: {t.user_attrs.get('cagr')}")
#     print(f"  Max Drawdown: {t.user_attrs.get('max_drawdown')}")
#     print(f"  Score Components: {t.user_attrs.get('score_components')}")


In [ ]:
start_date = "1945"
end_date = "2025"
portfolios = {}
short_names = []

# portfolios['50%'] = p_base.loc[start_date:end_date]
# short_names.append('50%')
# portfolios['50%+G'] = p_g.loc[start_date:end_date]
# short_names.append('50%+G')
# portfolios['50%+N'] = p_n.loc[start_date:end_date]
# short_names.append('50%+N')
# portfolios['50%+NG'] = p_ng.loc[start_date:end_date]
# short_names.append('50%+NG')

# portfolios['50%+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('50%+MA')
# portfolios['50%+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('50%+N+MA')
# portfolios['50%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# short_names.append('50%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
portfolios["S&P500"] = p_sp500.loc[start_date:end_date]
short_names.append("S&P500")
portfolios["2x S&P500"] = p_2x_sp500.loc[start_date:end_date]
short_names.append("2x S&P500")
portfolios["2x S&P500 (MA)"] = p_2x_sp500_ma.loc[start_date:end_date]
short_names.append("2x S&P500 (MA)")
portfolios["2x NS100 (MA)"] = p_2x_ndx100_ma.loc[start_date:end_date]
short_names.append("2x NS100 (MA)")

portfolios["2x NS100+S&P500 (MA)"] = p_2x_ndx100_2x_sp500_ma.loc[start_date:end_date]
short_names.append("2x NS100&SP500 (MA)")
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

for v in portfolios.values():
    v = normalize_df(v, start_value=1000)

compare_portfolios(
    portfolios,
    short_names=short_names,
    details=True,
)

## 65% Portfolio

In [ ]:
# p_base = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=65),
#         '1x_ltt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=65),
#         '1x_ltt_eu': dict(dist=26.25),
#         '1x_gold_eu': dict(dist=8.75),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=55.25),
#         '2x_ndx100_eu': dict(dist=9.75),
#         '1x_ltt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=42.5),
# #        '2x_ndx100_eu': dict(dist=7.5),
# #        '1x_ltt_eu': dict(dist=37.5),
# #        '1x_gold_eu': dict(dist=12.5),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=65, ma=290, ma_asset="1x_sp500_eu"),
#         '1x_ltt_eu': dict(dist=35, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=55.25, ma=290, ma_asset="1x_sp500_eu"),
#         '2x_ndx100_eu': dict(dist=9.75, ma=310, ma_asset="1x_ndx100_eu"),
#         '1x_ltt_eu': dict(dist=35, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng_ma = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=42.5, ma=290, ma_asset="1x_sp500_eu"),
# #        '2x_ndx100_eu': dict(dist=7.5, ma=310, ma_asset="1x_ndx100_eu"),
# #        '1x_ltt_eu': dict(dist=37.5, ma=130, ma_asset="1x_ltt_eu"),
# #        '1x_gold_eu': dict(dist=12.5, ma=400, ma_asset="1x_gold_eu"),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['65%'] = p_base.loc[start_date:end_date]
# short_names.append('65%')
# portfolios['65%+G'] = p_g.loc[start_date:end_date]
# short_names.append('65%+G')
# portfolios['65%+N'] = p_n.loc[start_date:end_date]
# short_names.append('65%+N')
# #portfolios['50%+NG'] = p_ng.loc[start_date:end_date]
# #short_names.append('65%+NG')

# portfolios['65%+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('65%+MA')
# portfolios['65%+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('65%+N+MA')
# #portfolios['65%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# #short_names.append('65%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## 80% Portfolio

In [ ]:
# p_base = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=80),
#         '1x_ltt_eu': dict(dist=20),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=80),
#         '1x_ltt_eu': dict(dist=15),
#         '1x_gold_eu': dict(dist=5),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68),
#         '2x_ndx100_eu': dict(dist=12),
#         '1x_ltt_eu': dict(dist=20),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68),
#         '2x_ndx100_eu': dict(dist=12),
#         '1x_ltt_eu': dict(dist=15),
#         '1x_gold_eu': dict(dist=5),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=80, ma=290, ma_asset="1x_sp500_eu"),
#         '1x_ltt_eu': dict(dist=20, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68, ma=290, ma_asset="1x_sp500_eu"),
#         '2x_ndx100_eu': dict(dist=12, ma=310, ma_asset="1x_ndx100_eu"),
#         '1x_ltt_eu': dict(dist=20, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68, ma=290, ma_asset="1x_sp500_eu"),
#         '2x_ndx100_eu': dict(dist=12, ma=310, ma_asset="1x_ndx100_eu"),
#         '1x_ltt_eu': dict(dist=15, ma=130, ma_asset="1x_ltt_eu"),
#         '1x_gold_eu': dict(dist=5, ma=400, ma_asset="1x_gold_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['80%'] = p_base.loc[start_date:end_date]
# short_names.append('80%')
# portfolios['80%+G'] = p_g.loc[start_date:end_date]
# short_names.append('80%+G')
# portfolios['80%+N'] = p_n.loc[start_date:end_date]
# short_names.append('80%+N')
# portfolios['80%+NG'] = p_ng.loc[start_date:end_date]
# short_names.append('80%+NG')

# portfolios['80%+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('80%+MA')
# portfolios['80%+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('80%+N+MA')
# portfolios['80%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# short_names.append('80%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## 65% (3x) Portfolio

In [ ]:
# p_base = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=65),
#         '3x_itt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=65),
#         '3x_itt_eu': dict(dist=26.25),
#         '1x_gold_eu': dict(dist=8.75),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=55.25),
#         '3x_ndx100_eu': dict(dist=9.75),
#         '3x_itt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=68),
# #        '2x_ndx100_eu': dict(dist=12),
# #        '1x_ltt_eu': dict(dist=15),
# #        '1x_gold_eu': dict(dist=5),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=65, ma=290, ma_asset="1x_sp500_eu"),
#         '3x_itt_eu': dict(dist=35, ma=70, ma_asset="1x_itt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=55.25, ma=290, ma_asset="1x_sp500_eu"),
#         '3x_ndx100_eu': dict(dist=9.75, ma=310, ma_asset="1x_ndx100_eu"),
#         '3x_itt_eu': dict(dist=35, ma=70, ma_asset="1x_itt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng_ma = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=68, ma=290, ma_asset="1x_sp500_eu"),
# #        '2x_ndx100_eu': dict(dist=12, ma=310, ma_asset="1x_ndx100_eu"),
# #        '1x_ltt_eu': dict(dist=15, ma=130, ma_asset="1x_ltt_eu"),
# #        '1x_gold_eu': dict(dist=5, ma=400, ma_asset="1x_gold_eu"),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['65% (3x)'] = p_base.loc[start_date:end_date]
# short_names.append('65% (3x)')
# portfolios['65%+G (3x)'] = p_g.loc[start_date:end_date]
# short_names.append('65%+G (3x)')
# portfolios['65%+N (3x)'] = p_n.loc[start_date:end_date]
# short_names.append('65%+N (3x)')
# #portfolios['80%+NG'] = p_ng.loc[start_date:end_date]
# #short_names.append('80%+NG')

# portfolios['65%+MA (3x)'] = p_base_ma.loc[start_date:end_date]
# short_names.append('65%+MA (3x)')
# portfolios['65%+N+MA (3x)'] = p_n_ma.loc[start_date:end_date]
# short_names.append('65%+N+MA (3x)')
# #portfolios['80%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# #short_names.append('80%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## HFEA

In [ ]:
# p_base = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=55),
#         '3x_ltt_us': dict(dist=45),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=55),
#         '3x_ltt_us': dict(dist=33.75),
#         '1x_gold_eu': dict(dist=11.25),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25),
#         '3x_ndx100_us': dict(dist=13.75),
#         '3x_ltt_us': dict(dist=45),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25),
#         '3x_ndx100_us': dict(dist=13.75),
#         '3x_ltt_us': dict(dist=38.25),
#         '1x_gold_eu': dict(dist=6.75),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=55, ma=290, ma_asset="1x_sp500_us"),
#         '3x_ltt_us': dict(dist=45, ma=130, ma_asset="1x_ltt_us"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25, ma=290, ma_asset="1x_sp500_us"),
#         '3x_ndx100_us': dict(dist=13.75, ma=310, ma_asset="1x_ndx100_us"),
#         '3x_ltt_us': dict(dist=45, ma=130, ma_asset="1x_ltt_us"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng_ma = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25, ma=290, ma_asset="1x_sp500_us"),
#         '3x_ndx100_us': dict(dist=13.75, ma=310, ma_asset="1x_ndx100_us"),
#         '3x_ltt_us': dict(dist=38.25, ma=130, ma_asset="1x_ltt_us"),
#         '1x_gold_eu': dict(dist=6.75, ma=400, ma_asset="1x_gold_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['HFEA'] = p_base.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['HFEA+G'] = p_g.loc[start_date:end_date]
# short_names.append('HFEA+G')
# portfolios['HFEA+N'] = p_n.loc[start_date:end_date]
# short_names.append('HFEA+N')
# portfolios['HFEA+NG'] = p_ng.loc[start_date:end_date]
# short_names.append('HFEA+NG')

# portfolios['HFEA+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('HFEA+MA')
# portfolios['HFEA+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('HFEA+N+MA')
# portfolios['HFEA+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# short_names.append('HFEA+NG+MA')

# #portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# #short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['3x S&P500 (MA)'] = p_3x_sp500_ma.loc[start_date:end_date]
# short_names.append('3x S&P500 (MA)')
# portfolios['2x NDX100 (MA)'] = p_2x_ndx100_ma.loc[start_date:end_date]
# short_names.append('2x NDX100 (MA)')
# portfolios['3x NDX100 (MA)'] = p_3x_ndx100_ma.loc[start_date:end_date]
# short_names.append('3x NDX100 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## Comparison (Non-Tax)

In [ ]:
allocations = [
    (
        "2x S&P 500",
        ("2x_sp500_eu", 100),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 0),
        ("1x_gold_eu", 0),
    ),
    (
        "50%",
        ("2x_sp500_eu", 50),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 50),
        ("1x_gold_eu", 0),
    ),
    (
        "50%+G",
        ("2x_sp500_eu", 50),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 37.5),
        ("1x_gold_eu", 12.5),
    ),
    (
        "50%+N",
        ("2x_sp500_eu", 42.5),
        ("2x_ndx100_eu", 7.5),
        ("1x_ltt_eu", 50),
        ("1x_gold_eu", 0),
    ),
    (
        "50%+NG",
        ("2x_sp500_eu", 42.5),
        ("2x_ndx100_eu", 7.5),
        ("1x_ltt_eu", 37.50),
        ("1x_gold_eu", 12.5),
    ),
    (
        "65%",
        ("2x_sp500_eu", 65),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "65%+G",
        ("2x_sp500_eu", 65),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 26.25),
        ("1x_gold_eu", 8.75),
    ),
    (
        "65%+N",
        ("2x_sp500_eu", 55.25),
        ("2x_ndx100_eu", 9.75),
        ("1x_ltt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "80%",
        ("2x_sp500_eu", 80),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 20),
        ("1x_gold_eu", 0),
    ),
    (
        "80%+G",
        ("2x_sp500_eu", 80),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 15),
        ("1x_gold_eu", 5),
    ),
    (
        "80%+N",
        ("2x_sp500_eu", 68),
        ("2x_ndx100_eu", 12),
        ("1x_ltt_eu", 20),
        ("1x_gold_eu", 0),
    ),
    (
        "80%+NG",
        ("2x_sp500_eu", 68),
        ("2x_ndx100_eu", 12),
        ("1x_ltt_eu", 15),
        ("1x_gold_eu", 5),
    ),
    (
        "65% (3x)",
        ("3x_sp500_eu", 65),
        ("3x_ndx100_eu", 0),
        ("3x_itt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "65%+G (3x)",
        ("3x_sp500_eu", 65),
        ("3x_ndx100_eu", 0),
        ("3x_itt_eu", 26.25),
        ("1x_gold_eu", 8.75),
    ),
    (
        "65%+N (3x)",
        ("3x_sp500_eu", 55.25),
        ("3x_ndx100_eu", 9.75),
        ("3x_itt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "HFEA",
        ("3x_sp500_us", 55),
        ("3x_ndx100_us", 0),
        ("3x_ltt_us", 45),
        ("1x_gold_us", 0),
    ),
    (
        "HFEA+G",
        ("3x_sp500_us", 55),
        ("3x_ndx100_us", 0),
        ("3x_ltt_us", 33.75),
        ("1x_gold_us", 11.25),
    ),
    (
        "HFEA+N",
        ("3x_sp500_us", 41.25),
        ("3x_ndx100_us", 13.75),
        ("3x_ltt_us", 45),
        ("1x_gold_us", 0),
    ),
    (
        "HFEA+NG",
        ("3x_sp500_us", 41.25),
        ("3x_ndx100_us", 13.75),
        ("3x_ltt_us", 38.25),
        ("1x_gold_us", 6.75),
    ),
]


short_names = []
portfolios = {}
for a in allocations:
    name = a[0]
    print(f"Calculate: {name}")
    short_names.append(name)
    portfolios[name] = Portfolio(
        {
            a[1][0]: a[1][1],
            a[2][0]: a[2][1],
            a[3][0]: a[3][1],
            a[4][0]: a[4][1],
        },
        start_value=1000,
        rebalancing=relativedelta(months=3),
        rebalancing_offset=relativedelta(days=-6),
    ).backtest(etfs)


portfolios["S&P500"] = p_sp500
short_names.append("S&P500")
portfolios["P"] = p_pari
short_names.append("P")

compare_portfolios(
    portfolios,
    short_names=short_names,
    details=True,
)